# 29 — FinancialBERT tez düzeltmeleri

Bu notebook eğitim yapmaz. 28A checkpoint'lerini ve 28B'nin kayıtlı target/consensus tahminlerini kullanır.

Düzeltmeler:
- Gerçek hazır sentiment checkpoint'i: `ahmedrachid/FinancialBERT-Sentiment-Analysis`
- Hazır / target / consensus için üç doğrudan McNemar karşılaştırması
- Holm-Bonferroni düzeltmesi
- Çok küçük p-değerlerinin `0.0` yerine raporlanabilir gösterimi


In [1]:
from pathlib import Path
import json, shutil, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix
from thesis_utils import paths
from thesis_utils.statistics import mcnemar_test, holm_adjust

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
READY_MODEL_NAME = 'ahmedrachid/FinancialBERT-Sentiment-Analysis'
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}; LABELS=list(LABEL2ID)
OLD_ROOT = paths.MODEL_RESULTS_ROOT/'financialbert_full_pipeline_financialbert_seed42_v2'
NEW_ROOT = paths.MODEL_RESULTS_ROOT/'financialbert_thesis_corrected_v1'
NEW_ROOT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE); print('Ready model:', READY_MODEL_NAME); print('Output:', NEW_ROOT)


Device: cpu
Ready model: ahmedrachid/FinancialBERT-Sentiment-Analysis
Output: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\outputs\finetuned_model_results\financialbert_thesis_corrected_v1


In [2]:
def norm_label(x):
    s=str(x).lower().strip().replace(' ','_')
    return {'label_0':'positive','label_1':'negative','label_2':'neutral','pos':'positive','neg':'negative','neu':'neutral'}.get(s,s)

def evaluate_frame(df, probs, name, outdir):
    y=df['label_id'].astype(int).to_numpy(); pred=probs.argmax(1); out=df.reset_index(drop=True).copy()
    out['gold_label']=[ID2LABEL[int(x)] for x in y]; out['predicted_label']=[ID2LABEL[int(x)] for x in pred]; out['correct']=y==pred; out['confidence']=probs.max(1)
    for i,label in ID2LABEL.items(): out[f'prob_{label}']=probs[:,i]
    p,r,f1,_=precision_recall_fscore_support(y,pred,labels=[0,1,2],zero_division=0)
    m={'stage':name,'n_eval':len(y),'accuracy':accuracy_score(y,pred),'precision_macro':p.mean(),'recall_macro':r.mean(),'f1_macro':f1.mean(),'f1_weighted':f1_score(y,pred,average='weighted',zero_division=0),'f1_negative':f1[0],'f1_neutral':f1[1],'f1_positive':f1[2],'model_name':READY_MODEL_NAME}
    outdir.mkdir(parents=True,exist_ok=True); (outdir/f'{name}_metrics.json').write_text(json.dumps(m,indent=2),encoding='utf-8'); out.to_csv(outdir/f'{name}_predictions.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(classification_report(y,pred,labels=[0,1,2],target_names=LABELS,output_dict=True,zero_division=0)).T.to_csv(outdir/f'{name}_classification_report.csv')
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1,2]),index=[f'true_{x}' for x in LABELS],columns=[f'pred_{x}' for x in LABELS]).to_csv(outdir/f'{name}_confusion_matrix.csv')
    return m,out

@torch.no_grad()
def predict_model(model, tok, texts, class_order=None, batch_size=64):
    rows=[]
    for i in tqdm(range(0,len(texts),batch_size),desc='ready sentiment predicting'):
        enc=tok(texts[i:i+batch_size],padding=True,truncation=True,max_length=512,return_tensors='pt')
        enc={k:v.to(DEVICE) for k,v in enc.items()}; logits=model(**enc).logits
        if class_order is not None: logits=logits[:,class_order]
        rows.append(torch.softmax(logits,dim=-1).cpu().numpy())
    return np.concatenate(rows)

ready_tok=AutoTokenizer.from_pretrained(READY_MODEL_NAME,use_fast=True)
ready_model=AutoModelForSequenceClassification.from_pretrained(READY_MODEL_NAME).to(DEVICE).eval()
raw={int(k):norm_label(v) for k,v in ready_model.config.id2label.items()}
if set(raw.values()) != set(LABEL2ID): raise ValueError(f'Ready model labels unexpected: {raw}')
READY_ORDER=[next(i for i,label in raw.items() if label==wanted) for wanted in LABELS]
print('Ready id2label:',raw,'reorder:',READY_ORDER)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Ready id2label: {0: 'negative', 1: 'neutral', 2: 'positive'} reorder: [0, 1, 2]


In [3]:
def copy_existing(stage, filename):
    src=OLD_ROOT/stage/filename; dst=NEW_ROOT/stage/filename; dst.parent.mkdir(parents=True,exist_ok=True)
    if not src.exists(): raise FileNotFoundError(src)
    shutil.copy2(src,dst); return pd.read_csv(src)

# İç test: target ve consensus tahminleri yeniden eğitilmeden aynen alınır.
target_df=copy_existing('target_dataset','target_internal_test_predictions.csv')
consensus_df=copy_existing('consensus','consensus_internal_test_predictions.csv')


In [4]:
# Doğru hazır FinancialBERT sentiment baseline — old 0.2854 sonucu kullanılmaz.
src=OLD_ROOT/'ready_baseline'/'ready_baseline_predictions.csv'; ready_df=pd.read_csv(src)
ready_df['label_id']=ready_df['label_id'].astype(int)
ready_probs=predict_model(ready_model,ready_tok,ready_df['input_text'].astype(str).tolist(),class_order=READY_ORDER)
ready_metrics,ready_pred=evaluate_frame(ready_df,ready_probs,'ready_baseline_corrected',NEW_ROOT/'ready_baseline')
print('Corrected ready baseline:',ready_metrics)

# Target ve consensus iç-test çıktılarının metric dosyalarını da yeni köke kopyala.
for stage,name in [('target_dataset','target_internal_test'),('consensus','consensus_internal_test')]:
    for suffix in ['_metrics.json','_classification_report.csv','_confusion_matrix.csv']:
        src=OLD_ROOT/stage/f'{name}{suffix}'; dst=NEW_ROOT/stage/f'{name}{suffix}'
        if src.exists(): dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)


ready sentiment predicting:   0%|          | 0/38 [00:00<?, ?it/s]

Corrected ready baseline: {'stage': 'ready_baseline_corrected', 'n_eval': 2386, 'accuracy': 0.7518860016764459, 'precision_macro': np.float64(0.7153364330536295), 'recall_macro': np.float64(0.611254503823247), 'f1_macro': np.float64(0.6456921919196275), 'f1_weighted': 0.7364249778713653, 'f1_negative': np.float64(0.5026548672566372), 'f1_neutral': np.float64(0.8318425760286225), 'f1_positive': np.float64(0.6025791324736225), 'model_name': 'ahmedrachid/FinancialBERT-Sentiment-Analysis'}


In [5]:
DATASETS=['sp500_external','synthetic_external','reuters_external']
MODEL_KEYS=['ready','target','consensus']

# Target/consensus dış tahminleri mevcut kayıtlarından al; ready dış tahminlerini doğru checkpoint ile yeniden üret.
for dataset in DATASETS:
    for key in ['target','consensus']:
        src=OLD_ROOT/'external'/key/f'{key}_{dataset}_predictions.csv'
        dst=NEW_ROOT/'external'/key/f'{key}_{dataset}_predictions.csv'; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
        for suffix in ['_metrics.json','_classification_report.csv','_confusion_matrix.csv']:
            s=OLD_ROOT/'external'/key/f'{key}_{dataset}{suffix}'; d=NEW_ROOT/'external'/key/f'{key}_{dataset}{suffix}'
            if s.exists(): shutil.copy2(s,d)
    src=OLD_ROOT/'external'/'ready'/f'ready_{dataset}_predictions.csv'; raw_df=pd.read_csv(src); raw_df['label_id']=raw_df['label_id'].astype(int)
    probs=predict_model(ready_model,ready_tok,raw_df['input_text'].astype(str).tolist(),class_order=READY_ORDER)
    evaluate_frame(raw_df,probs,f'ready_{dataset}',NEW_ROOT/'external'/'ready')
print('Corrected ready external tests completed.')


ready sentiment predicting:   0%|          | 0/17 [00:00<?, ?it/s]

ready sentiment predicting:   0%|          | 0/47 [00:00<?, ?it/s]

ready sentiment predicting:   0%|          | 0/79 [00:00<?, ?it/s]

Corrected ready external tests completed.


In [6]:
def format_p(p):
    p=float(p)
    return '<1e-300' if p<=0 else f'{p:.6g}'

rows=[]
for dataset in DATASETS:
    frames={key:pd.read_csv(NEW_ROOT/'external'/key/f'{key}_{dataset}_predictions.csv') for key in MODEL_KEYS}
    for a,b in [('ready','target'),('ready','consensus'),('target','consensus')]:
        left,right=frames[a],frames[b]
        if not left['sample_id'].astype(str).equals(right['sample_id'].astype(str)): raise ValueError(f'{dataset}: sample_id mismatch for {a}/{b}')
        ca=(left.gold_label==left.predicted_label).to_numpy(); cb=(right.gold_label==right.predicted_label).to_numpy()
        result=mcnemar_test(ca,cb)
        rows.append({'dataset':dataset,'reference':a,'comparison':b,'n':len(left),'reference_accuracy':float(ca.mean()),'comparison_accuracy':float(cb.mean()),**result})
stats=pd.DataFrame(rows)
stats['holm_p_value']=holm_adjust(stats['p_value'].to_numpy())
stats['p_value_report']=stats['p_value'].map(format_p)
stats['holm_p_value_report']=stats['holm_p_value'].map(format_p)
stats.to_csv(NEW_ROOT/'financialbert_mcnemar_comparisons_holm.csv',index=False,encoding='utf-8-sig')
display(stats[['dataset','reference','comparison','discordant','p_value_report','holm_p_value_report','method']])


,dataset,reference,comparison,discordant,p_value_report,holm_p_value_report,method
0,sp500_external,ready,target,284,8.6186e-09,3.44744e-08,chi_square_continuity_corrected
1,sp500_external,ready,consensus,303,2.66427e-11,1.33213e-10,chi_square_continuity_corrected
2,sp500_external,target,consensus,187,0.188077,0.376153,chi_square_continuity_corrected
3,synthetic_external,ready,target,85,3.87205e-12,2.71043e-11,chi_square_continuity_corrected
4,synthetic_external,ready,consensus,80,9.10359e-12,5.46215e-11,chi_square_continuity_corrected
5,synthetic_external,target,consensus,29,0.710347,0.710347,chi_square_continuity_corrected
6,reuters_external,ready,target,800,1.65226e-24,1.32181e-23,chi_square_continuity_corrected
7,reuters_external,ready,consensus,839,1.82853e-35,1.64568e-34,chi_square_continuity_corrected
8,reuters_external,target,consensus,507,0.00187843,0.00563529,chi_square_continuity_corrected


## Tez kullanım notu

- `ready_baseline_corrected` gerçek hazır FinancialBERT sentiment checkpoint'inden üretilmiştir.
- Target ve consensus modelleri yeniden eğitilmemiş, kayıtlı checkpoint/prediction çıktıları kullanılmıştır.
- McNemar karşılaştırmaları tüm üç model çifti için yapılmıştır: ready–target, ready–consensus, target–consensus.
- Holm düzeltilmiş sonuçlar `financialbert_mcnemar_comparisons_holm.csv` dosyasındadır.
